In [ ]:
import pandas as pd
import numpy as np

df = pd.read_excel("skeleton.xlsx")

df.head()

In [ ]:
sequence_id = "P3_S01_R4"

seq_df = df[df["kode_video"] == sequence_id].copy()

print(seq_df.shape)
seq_df.head()

In [ ]:
import numpy as np
from scipy.interpolate import interp1d

def missing_keypoint_reconstruction(origin_input_data):
    """
    Missing keypoint reconstruction using temporal interpolation.

    Parameters
    ----------
    origin_input_data : Tensor
        Skeleton sequence with shape (T, K, C).

    Returns
    -------
    Tensor
        Reconstructed skeleton sequence.
    """
    # Buat salinan untuk hasil rekonstruksi
    result = origin_input_data.clone()

    # Ekstrak koordinat x dan y
    kp_xy = result[:, :, 0:2].cpu().numpy().astype(np.float32)
    # T = jumlah frame, K = jumlah keypoint
    T, K, _ = kp_xy.shape

    for k in range(K):

        coords = kp_xy[:, k, :]  # (T, 2)

        # Keypoint dianggap missing jika x == 0 dan y == 0
        valid_mask = ~(
            (coords[:, 0] == 0) &
            (coords[:, 1] == 0)
        )

        valid_idx = np.where(valid_mask)[0]

        # Semua frame missing
        if len(valid_idx) == 0:
            continue

        for t in range(T):

            # Skip jika valid
            if valid_mask[t]:
                continue

            prev_arr = valid_idx[valid_idx < t]
            next_arr = valid_idx[valid_idx > t]

            # Interpolasi linier
            if len(prev_arr) and len(next_arr):

                p = prev_arr[-1]
                n = next_arr[0]

                alpha = (t - p) / (n - p)

                coords[t] = (
                    (1 - alpha) * coords[p] +
                    alpha * coords[n]
                )

            # Gunakan frame sebelumnya
            elif len(prev_arr):
                coords[t] = coords[prev_arr[-1]]

            # Gunakan frame berikutnya
            elif len(next_arr):
                coords[t] = coords[next_arr[0]]

        kp_xy[:, k, :] = coords

    # Masukkan kembali hasil rekonstruksi
    result[:, :, 0:2] = torch.from_numpy(kp_xy).to(
        device=result.device,
        dtype=result.dtype
    )

    return result